1. Decisions setup
- Prepare the Decisions API module and test data.
- Inspect the article list and sentence-level input.


In [125]:
# -*- coding: utf-8 -*-
import importlib
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r'C:\work\ner')
TEST_DATA_PATH = PROJECT_ROOT / 'data' / 'test' / 'test_data_v2'
RULE_DATA_PATH = PROJECT_ROOT / 'data' / 'filter' / 'rule_detect_data'
DATA_PATH = TEST_DATA_PATH
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.ner_review.decision.decision as decision_module
decision_module = importlib.reload(decision_module)

print(f"data path: {DATA_PATH}")

data path: C:\work\ner\data\test\test_data_v2


In [126]:
# Inspect the article list
press = '강원일보'
articles = decision_module.list_articles(DATA_PATH, press=press)
display(articles.head(20))

# Inspect the first article
article_index = 0
selected_article = articles.iloc[article_index]
article = decision_module.load_documents(selected_article['file'])[int(selected_article['article_index'])]
print('doc_id:', article['id'])
print('sentence count:', len(decision_module.prepare_document(article)[1]))


,press,file,article_index,doc_id,title,sentence_count
0,강원일보,C:\work\ner\data\test\test_data_v2\강원일보.json,0,NLRW2600000001.1,강원일보 2025년 기사,14
1,강원일보,C:\work\ner\data\test\test_data_v2\강원일보.json,1,NLRW2600000001.2,강원일보 2025년 기사,19
2,강원일보,C:\work\ner\data\test\test_data_v2\강원일보.json,2,NLRW2600000001.6,강원일보 2025년 기사,13
3,강원일보,C:\work\ner\data\test\test_data_v2\강원일보.json,3,NLRW2600000001.19,강원일보 2025년 기사,12
4,강원일보,C:\work\ner\data\test\test_data_v2\강원일보.json,4,NLRW2600000001.21,강원일보 2025년 기사,13


doc_id: NLRW2600000001.1
sentence count: 14


2. Run Decisions
- Run 
un_pipeline to generate sentence-level Decisions results.
- Flatten nswers to inspect each rule choice and probability.


In [ ]:
MODEL_OVERRIDE = None

# sentence_id = decision_module.prepare_document(article)[1][0]['id']
# results = decision_module.run_pipeline(DATA_PATH, press=press, article_index=article_index, sentence_ids=[sentence_id], model=MODEL_OVERRIDE)

results = decision_module.run_pipeline(DATA_PATH, press=press, article_index=article_index, model=MODEL_OVERRIDE)

result_frame = pd.DataFrame(results)
display(result_frame)


In [ ]:
# results = decision_module.run_pipeline(DATA_PATH, model=MODEL_OVERRIDE)

In [ ]:
result_frame = pd.DataFrame(results)

In [ ]:
completed_rows = result_frame[result_frame["answers"].map(lambda value: isinstance(value, list) and len(value) > 0)]
answer_counts = completed_rows["answers"].map(len)
answer_frame = completed_rows.loc[completed_rows.index.repeat(answer_counts)].drop(columns="answers").reset_index(drop=True)
answer_records = [answer for answers in completed_rows["answers"] for answer in answers]
answer_frame = pd.concat([answer_frame, pd.json_normalize(answer_records)], axis=1)

In [ ]:
answer_frame["rule"] = answer_frame["name"].str.rsplit("__", n=1).str[-1]
answer_frame["name"] = answer_frame["rule"]

sentence_frame = pd.DataFrame(decision_module.prepare_document(article)[1])[["id", "form"]].rename(columns={"id": "sentence_id", "form": "sentence"})
answer_frame = sentence_frame.merge(answer_frame, on="sentence_id", how="right")

In [ ]:
answer_frame["probability"] = answer_frame.apply(
    lambda row: next(
        (item.get("probability") for item in row.get("raw.probabilities", []) if item.get("value") == row["choice"]),
        None,
    ),
    axis=1,
)


In [ ]:
display(answer_frame[["sentence_id", "sentence", "name", "type", "choice", "value", "probability"]])

In [ ]:
result_wide = answer_frame.pivot(
    index="sentence_id",
    columns="rule",
    values=["choice", "probability"],
)

In [ ]:
result_wide.columns = [
    f"{name}_{field}"
    for field, name in result_wide.columns
]

result_wide = result_wide.reset_index()

In [ ]:
result_wide

In [129]:
catalog = decision_module.build_catalog(DATA_PATH)
catalog

,press,file,format,article_count,sentence_count
0,강원일보,C:\work\ner\data\test\test_data_v2\강원일보.json,.json,5,71
1,경기일보,C:\work\ner\data\test\test_data_v2\경기일보.json,.json,2,27
2,경향신문,C:\work\ner\data\test\test_data_v2\경향신문.json,.json,6,106
3,노컷뉴스,C:\work\ner\data\test\test_data_v2\노컷뉴스.json,.json,4,91
4,뉴스1,C:\work\ner\data\test\test_data_v2\뉴스1.json,.json,3,35
5,뉴스핌,C:\work\ner\data\test\test_data_v2\뉴스핌.json,.json,2,41
6,뉴시스,C:\work\ner\data\test\test_data_v2\뉴시스.json,.json,3,30
7,디지털타임스,C:\work\ner\data\test\test_data_v2\디지털타임스.json,.json,2,36
8,마이데일리,C:\work\ner\data\test\test_data_v2\마이데일리.json,.json,1,19
9,부산일보,C:\work\ner\data\test\test_data_v2\부산일보.json,.json,3,35


In [130]:
from datetime import datetime

press_results = []
press_list = catalog["press"].dropna().drop_duplicates().tolist()

In [ ]:
# decision 랭스미스 20261112 (response는 20261111 여기에 그 전 테스트까지만 포함되어잇음)

In [133]:
for press_index, current_press in enumerate(press_list, start=1):
    run_id = f"run_{datetime.now():%Y%m%d_%H%M%S}"
    output_root = PROJECT_ROOT / "outputs" / "decision" / run_id
    print(f"[{press_index}/{len(press_list)}] {current_press}")
    press_results.extend(
        decision_module.run_pipeline(
            DATA_PATH,
            output_root=output_root,
            press=current_press,
            model=MODEL_OVERRIDE,
        )
    )

press_result_frame = pd.DataFrame(press_results)
display(press_result_frame)

[1/18] 강원일보
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.1.1 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.2.1 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.3.1 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.1 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.2 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.3 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.4 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.5 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.6 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.7 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.8 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.9 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.10 / completed
강원일보 / NLRW2600000001.1 / NLRW2600000001.1.4.11 / completed
강원일보 / NLRW2600000001.2 / NLRW2600000001.2.1.1 / completed
강원일보 / NLRW2600000001.2 / NLRW2600000001.2.2.1 / completed
강원일보 / NLRW2600000001.2 / NLRW2600000001.2

,run_id,press,doc_id,sentence_id,status,answers
0,run_20261112_184507,강원일보,NLRW2600000001.1,NLRW2600000001.1.1.1,completed,"[{'name': 'LLM_E001', 'type': 'choice', 'choic..."
1,run_20261112_184507,강원일보,NLRW2600000001.1,NLRW2600000001.1.2.1,completed,"[{'name': 'LLM_E001', 'type': 'choice', 'choic..."
2,run_20261112_184507,강원일보,NLRW2600000001.1,NLRW2600000001.1.3.1,completed,"[{'name': 'LLM_E001', 'type': 'choice', 'choic..."
3,run_20261112_184507,강원일보,NLRW2600000001.1,NLRW2600000001.1.4.1,completed,"[{'name': 'LLM_E001', 'type': 'choice', 'choic..."
4,run_20261112_184507,강원일보,NLRW2600000001.1,NLRW2600000001.1.4.2,completed,"[{'name': 'LLM_E001', 'type': 'choice', 'choic..."
...,...,...,...,...,...,...
794,run_20261112_184957,충청일보,NLRW2600000004.26,NLRW2600000004.26.10.1,completed,"[{'name': 'LLM_E001', 'type': 'choice', 'choic..."
795,run_20261112_184957,충청일보,NLRW2600000004.26,NLRW2600000004.26.11.1,completed,"[{'name': 'LLM_E001', 'type': 'choice', 'choic..."
796,run_20261112_184957,충청일보,NLRW2600000004.26,NLRW2600000004.26.12.1,completed,"[{'name': 'LLM_E001', 'type': 'choice', 'choic..."
797,run_20261112_184957,충청일보,NLRW2600000004.26,NLRW2600000004.26.13.1,completed,"[{'name': 'LLM_E001', 'type': 'choice', 'choic..."


In [135]:
# press_result_frame["answers"][60]

In [ ]:
# result_frame = press_result_frame.copy()

In [ ]:
# completed_rows = result_frame[result_frame["answers"].map(lambda value: isinstance(value, list) and len(value) > 0)]
# answer_counts = completed_rows["answers"].map(len)
# answer_frame = completed_rows.loc[completed_rows.index.repeat(answer_counts)].drop(columns="answers").reset_index(drop=True)
# answer_records = [answer for answers in completed_rows["answers"] for answer in answers]
# answer_frame = pd.concat([answer_frame, pd.json_normalize(answer_records)], axis=1)

# answer_frame["rule"] = answer_frame["name"].str.rsplit("__", n=1).str[-1]
# answer_frame["name"] = answer_frame["rule"]

# sentence_frame = pd.DataFrame(decision_module.prepare_document(article)[1])[["id", "form"]].rename(columns={"id": "sentence_id", "form": "sentence"})
# answer_frame = sentence_frame.merge(answer_frame, on="sentence_id", how="right")

# answer_frame["probability"] = answer_frame.apply(
#     lambda row: next(
#         (item.get("probability") for item in row.get("raw.probabilities", []) if item.get("value") == row["choice"]),
#         None,
#     ),
#     axis=1,
# )

# result_wide = answer_frame.pivot(
#     index="sentence_id",
#     columns="rule",
#     values=["choice", "probability"],
# )

# result_wide.columns = [
#     f"{name}_{field}"
#     for field, name in result_wide.columns
# ]

# result_wide = result_wide.reset_index()

In [ ]:
# result_wide
# path = PROJECT_ROOT / "outputs" / "decision" / "run_20261112_171906" / press
# result_wide.to_csv(path, index=False, encoding="utf-8-sig")


---

In [140]:
# sentence-level rule export
expected_rules = ["LLM_E001", "LLM_S001", "LLM_E002", "LLM_E003"]
completed_rows = press_result_frame[press_result_frame["answers"].map(lambda value: isinstance(value, list))].copy()

sentence_rule_rows = []
for _, row in completed_rows.iterrows():
    rule_values = {rule: {"choice": pd.NA, "probability": pd.NA, "executed": False} for rule in expected_rules}
    for answer in row["answers"]:
        rule = str(answer.get("name", "")).rsplit("__", 1)[-1]
        if rule not in rule_values:
            continue
        raw = answer.get("raw") or {}
        probability = next((item.get("probability") for item in raw.get("probabilities", []) if item.get("value") == answer.get("choice")), pd.NA)
        rule_values[rule] = {"choice": answer.get("choice", pd.NA), "probability": probability, "executed": True}
    output_row = {column: row.get(column, pd.NA) for column in ["run_id", "press", "doc_id", "sentence_id", "status"]}
    for rule in expected_rules:
        output_row[f"{rule}_choice"] = rule_values[rule]["choice"]
        output_row[f"{rule}_probability"] = rule_values[rule]["probability"]
        output_row[f"{rule}_executed"] = rule_values[rule]["executed"]
    sentence_rule_rows.append(output_row)

sentence_rule_result_frame = pd.DataFrame(sentence_rule_rows).merge(sentence_frame[["sentence_id", "sentence"]].drop_duplicates(), on="sentence_id", how="left")

In [141]:
output_path = PROJECT_ROOT / "outputs" / "decision" / "sentence_rule_results.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
sentence_rule_result_frame.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"{output_path}")
display(sentence_rule_result_frame)

C:\work\ner\outputs\decision\sentence_rule_results.csv


,run_id,press,doc_id,sentence_id,status,LLM_E001_choice,LLM_E001_probability,LLM_E001_executed,LLM_S001_choice,LLM_S001_probability,LLM_S001_executed,LLM_E002_choice,LLM_E002_probability,LLM_E002_executed,LLM_E003_choice,LLM_E003_probability,LLM_E003_executed,sentence
0,run_20261112_184507,강원일보,NLRW2600000001.1,NLRW2600000001.1.1.1,completed,True,0.92,True,False,0.99,True,<NA>,<NA>,False,<NA>,<NA>,False,"<이재명: PS>, <강원도: LC>에 어떤 내용 공약했나"
1,run_20261112_184507,강원일보,NLRW2600000001.1,NLRW2600000001.1.2.1,completed,True,0.92,True,False,0.99,True,<NA>,<NA>,False,<NA>,<NA>,False,특별한 희생에 상응하는 특별한 보상 강조하며 남북 평화경제의 중심지로 만들겠다고 약속
2,run_20261112_184507,강원일보,NLRW2600000001.1,NLRW2600000001.1.3.1,completed,True,0.92,True,False,0.99,True,<NA>,<NA>,False,<NA>,<NA>,False,"미래산업 경쟁력 육성, <강원자치도: LC> 권한·자율성 확대, 사통팔달 <강원: ..."
3,run_20261112_184507,강원일보,NLRW2600000001.1,NLRW2600000001.1.4.1,completed,True,0.92,True,False,0.99,True,<NA>,<NA>,False,<NA>,<NA>,False,<이재명: PS> <더불어민주당: OG> 대선 후보는 <27일: DT> 합동연설회 ...
4,run_20261112_184507,강원일보,NLRW2600000001.1,NLRW2600000001.1.4.2,completed,True,0.92,True,False,0.99,True,<NA>,<NA>,False,<NA>,<NA>,False,"또 “<강원도: LC>는 전쟁위협과 접경지 규제로 오래 고통 받았지만, 이제 남북 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
794,run_20261112_184957,충청일보,NLRW2600000004.26,NLRW2600000004.26.10.1,completed,True,0.96,True,False,1.00,True,<NA>,<NA>,False,<NA>,<NA>,False,NaN
795,run_20261112_184957,충청일보,NLRW2600000004.26,NLRW2600000004.26.11.1,completed,True,0.96,True,False,1.00,True,<NA>,<NA>,False,<NA>,<NA>,False,NaN
796,run_20261112_184957,충청일보,NLRW2600000004.26,NLRW2600000004.26.12.1,completed,True,0.96,True,False,1.00,True,<NA>,<NA>,False,<NA>,<NA>,False,NaN
797,run_20261112_184957,충청일보,NLRW2600000004.26,NLRW2600000004.26.13.1,completed,True,0.96,True,False,1.00,True,<NA>,<NA>,False,<NA>,<NA>,False,NaN
